In [1]:
!pip install ./cacheflow-0.1.0-py3-none-any.whl

Processing ./cacheflow-0.1.0-py3-none-any.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 23.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1


In [2]:
from huggingface_hub import snapshot_download

model_name = "ibm-granite/granite-3.1-3b-a800m-instruct"
model_path = "./weights/granite/"

snapshot_download(model_name, local_dir=model_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

'/content/weights/granite'

In [3]:
import os
import json
from safetensors import safe_open
from safetensors.torch import save_file


def separate_safetensors_smart(input_path, output_directory):
    # 1. Create the output directory
    os.makedirs(output_directory, exist_ok=True)

    new_weight_map = {}
    total_size = 0
    files_to_process = []
    input_dir = os.path.dirname(input_path)

    # 2. Figure out if the input is a single file or an index.json
    if input_path.endswith(".json"):
        print(f"Detected index JSON. Reading {input_path}...")
        with open(input_path, "r") as f:
            index_data = json.load(f)

        # Look at the old weight map to find out what shard files exist
        old_weight_map = index_data.get("weight_map", {})

        # Get a list of unique shard files (e.g. model-00001-of-00003.safetensors)
        unique_shards = set(old_weight_map.values())

        # Build the full paths to those shards
        for shard in unique_shards:
            shard_path = os.path.join(input_dir, shard) if input_dir else shard
            files_to_process.append(shard_path)

        print(f"Found {len(files_to_process)} shard file(s) to unpack.")

    elif input_path.endswith(".safetensors"):
        print(f"Detected single safetensors file: {input_path}")
        files_to_process = [input_path]
    else:
        raise ValueError("Input file must be a .safetensors or .index.json file")

    # 3. Process every file we found
    tensor_count = 0

    for file_path in files_to_process:
        print(f"\nOpening {os.path.basename(file_path)}...")

        # Open lazily
        with safe_open(file_path, framework="pt", device="cpu") as f:
            keys = f.keys()
            for key in keys:
                tensor_count += 1

                # Extract only this specific layer into RAM
                tensor = f.get_tensor(key)

                # Create exact filename for this layer
                filename = f"{key}.safetensors"
                output_path = os.path.join(output_directory, filename)

                # Save it to the new folder
                save_file({key: tensor}, output_path)

                # Track size and map it for the new index
                file_size = os.path.getsize(output_path)
                total_size += file_size
                new_weight_map[key] = filename

                print(f"  Saved: {filename}")

    # 4. Generate the NEW model.safetensors.index.json
    print("\nAll layers separated. Generating new model.safetensors.index.json...")

    new_index_data = {
        "metadata": {"total_size": total_size},
        "weight_map": new_weight_map,
    }

    index_out_path = os.path.join(output_directory, "model.safetensors.index.json")

    with open(index_out_path, "w") as index_file:
        json.dump(new_index_data, index_file, indent=2)

    print(f"\n--- SUCCESS ---")
    print(f"Total separate tensors saved: {tensor_count}")
    print(f"Total model size: {total_size / (1024**3):.2f} GB")
    print(f"New index saved to: {index_out_path}")


# ==========================================
# CONFIGURATION - CHANGE THESE PATHS
# ==========================================

# Scenario A: Point this to a single large file
# INPUT_PATH = "model.safetensors"

# Scenario B: Point this to an existing index file
INPUT_PATH = "./weights/granite/model.safetensors.index.json"

# The folder where all the tiny files and the new index will be created
OUTPUT_DIR = "./weights/granite/"

separate_safetensors_smart(INPUT_PATH, OUTPUT_DIR)

Detected index JSON. Reading ./weights/granite/model.safetensors.index.json...
Found 2 shard file(s) to unpack.

Opening model-00002-of-00002.safetensors...
  Saved: model.layers.24.block_sparse_moe.input_linear.weight.safetensors
  Saved: model.layers.24.block_sparse_moe.output_linear.weight.safetensors
  Saved: model.layers.24.block_sparse_moe.router.layer.weight.safetensors
  Saved: model.layers.24.input_layernorm.weight.safetensors
  Saved: model.layers.24.post_attention_layernorm.weight.safetensors
  Saved: model.layers.25.block_sparse_moe.input_linear.weight.safetensors
  Saved: model.layers.25.block_sparse_moe.output_linear.weight.safetensors
  Saved: model.layers.25.block_sparse_moe.router.layer.weight.safetensors
  Saved: model.layers.25.input_layernorm.weight.safetensors
  Saved: model.layers.25.post_attention_layernorm.weight.safetensors
  Saved: model.layers.25.self_attn.k_proj.weight.safetensors
  Saved: model.layers.25.self_attn.o_proj.weight.safetensors
  Saved: model.la

In [4]:
from transformers import AutoTokenizer
import torch
from cacheflow import initialize_model


model_path = "./weights/granite/"
loaded = initialize_model(model_path)
print(loaded.device, loaded.dtype)


tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = loaded.config.pad_token_id

cuda torch.float16


In [6]:
torch.cuda.reset_peak_memory_stats()
question = "What is the capital of Bangladesh?"
prompt = f"Question: {question}\nAnswer:"
# prompt = f"write 10 line poem about python language"

result = loaded.generate_text(
    tokenizer=tokenizer,
    prompt=prompt,
    max_new_tokens=100,
    min_new_tokens=8,
    temperature=0.5,
    top_k=10,
    eos_token_id=loaded.config.eos_token_id,
    repetition_penalty=1.15,
    block_special_tokens=True,
    return_raw=True,
)

print("generated_token_ids:", result["token_ids"])
print("decoded_raw:", repr(result["raw_text"]))
print("decoded:", repr(result["text"]))

generated_token_ids: [475, 90, 13971, 203, 203, 3533, 225, 35, 44, 17492, 1147, 322, 10769, 328, 7000, 32, 203, 1318, 1256, 438, 24240, 2625, 551, 742, 94, 1525, 615, 30, 312, 1259, 29874, 627, 1603, 15282, 328, 30452, 48985, 32, 203, 203, 3533, 225, 36, 44, 22022, 322, 18926, 11297, 432, 322, 3191, 10769, 32, 203, 42620, 94, 1525, 615, 1401, 8019, 17268, 5205, 283, 3528, 28025, 2819, 8142, 7254, 372, 35566, 1578, 3404, 461, 22187, 1353, 316, 26707, 32, 886, 1550, 18926, 438, 475, 90, 13971, 30, 1510, 1597, 27493, 619, 322, 18926, 328, 225, 35, 43, 40, 38, 2685, 322]
decoded_raw: ' Dhaka\n\nStep 1: Identify the country in question.\nThe user is asking about Bangladesh, a sovereign state located in South Asia.\n\nStep 2: Determine the capital city of the specified country.\nBangladesh has three official capitals throughout its history due to political changes and administrative reorganizations. The current capital is Dhaka, which was established as the capital in 1964 after the'
decoded

In [7]:
print(
    "after warmup alloc/res:",
    torch.cuda.memory_allocated() / 1e9,
    torch.cuda.memory_reserved() / 1e9,
)
print("peak alloc:", torch.cuda.max_memory_allocated() / 1e9)
if torch.cuda.is_available():
    peak_mem = torch.cuda.max_memory_allocated() / (1024**2)
    curr_mem = torch.cuda.memory_allocated() / (1024**2)
    print(f"🔥 Peak VRAM Allocated:   {peak_mem:.2f} MB")
    print(f"   Current VRAM Allocated: {curr_mem:.2f} MB")

after warmup alloc/res: 0.920203776 5.077204992
peak alloc: 1.02247936
🔥 Peak VRAM Allocated:   975.11 MB
   Current VRAM Allocated: 877.57 MB
